# 读取增强版国家队因子：每个 sheet 与每个标的

- 从 **Data** 目录读取 `因子_国家队_增强版_pair_zscore.xlsx`
- 每个 **sheet** 对应一个因子类型（dt）
- 每个 **标的**（pair）为 sheet 内除「交易日期」外的列名
- 统一日期列名、规整类型，便于后续与 y 对齐

In [1]:
import os
import sys
import pandas as pd

_cwd = os.getcwd()
_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'factor_build' else _cwd
if _root not in sys.path:
    sys.path.insert(0, _root)
import config

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

DATA_DIR = config.get_data_dir()
DATE_COL = config.DATE_COL
FACTOR_FILE = getattr(config, 'AUGMENTED_FACTOR_FILENAME', '因子_国家队_增强版_pair_zscore.xlsx')
factor_path = os.path.join(DATA_DIR, FACTOR_FILE)
print('因子文件:', factor_path)
print('存在:', os.path.isfile(factor_path))

因子文件: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/Data/因子_国家队_增强版_pair_zscore.xlsx
存在: True


## 1. 读取所有 sheet

In [2]:
sheets_raw = pd.read_excel(factor_path, sheet_name=None)
EXCLUDE_SHEETS = getattr(config, 'EXCLUDE_SHEETS', [])
sheet_names = [
    n for n in sheets_raw.keys()
    if n != 'Target' and n not in EXCLUDE_SHEETS
]
print('Sheet 数量 (排除后):', len(sheet_names))
print('排除的因子:', EXCLUDE_SHEETS)
print('Sheet 列表:', sheet_names)


Sheet 数量 (排除后): 6
排除的因子: ['ETF申购赎回现金差额', 'amt', 'volume', 'amt_btin', 'iopv']
Sheet 列表: ['pct_chg', 'unit_total', 'netinflow', 'NAV_iopv_discount', 'turn', 'volume_btin']


## 2. 规整每个 sheet：统一日期列、列出标的

In [3]:
def normalize_date_col(df):
    """统一日期列名为 config.DATE_COL，并转为 date 类型。"""
    df = df.copy()
    if 'Date' in df.columns and DATE_COL not in df.columns:
        df = df.rename(columns={'Date': DATE_COL})
    if DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce').dt.date
    return df


def get_pair_cols(df):
    """标的 = 除日期列外的所有列。"""
    return [c for c in df.columns if c != DATE_COL]


# 规整后的每个 sheet：{ sheet_name: DataFrame }
augmented_sheets = {}
# 每个 (sheet, 标的) 的列表，便于按因子×标的遍历
sheet_pair_list = []

for name in sheet_names:
    df = sheets_raw[name].copy()
    df = normalize_date_col(df)
    augmented_sheets[name] = df
    pairs = get_pair_cols(df)
    for p in pairs:
        sheet_pair_list.append((name, p))

print('规整后 sheet 数:', len(augmented_sheets))
print('(sheet, 标的) 组合数:', len(sheet_pair_list))

规整后 sheet 数: 6
(sheet, 标的) 组合数: 45


## 2b. 存量因子转化为日变化量（unit_total → unit_chg）

对 `unit_total`（基金份额合计，存量）做**一阶差分**，得到 `unit_chg`（每日份额净变化）：
- 存量因子的 rolling std 很大（受长期趋势主导），同样金额的注入在 z-score 上不显著
- 变化量的 rolling std 仅为日频波动，注入脉冲能产生显著 z-score 跳变
- 经济含义：`unit_chg` ≈ `netinflow / NAV`，反映当日一级市场净申赎的份额量


In [4]:
# ── unit_total (存量) → unit_chg (日变化量) ─────────────────────────────────
# 存量因子的 rolling z-score 受趋势主导，信号弱。
# 一阶差分后再做 z-score，等价于衡量「今日申购/赎回的异常程度」，灵敏度更高。
# 注：unit_chg 与 netinflow 高度相关（差一个 NAV 系数），属互补冗余因子。

DIFF_SHEETS = {'unit_total': 'unit_chg'}  # 需要做一阶差分的 sheet 及其新名称

for old_name, new_name in DIFF_SHEETS.items():
    if old_name not in augmented_sheets:
        continue
    df = augmented_sheets[old_name].copy()
    pair_cols = get_pair_cols(df)
    for col in pair_cols:
        s = pd.to_numeric(df[col], errors='coerce').ffill()
        df[col] = s.diff()   # level → daily change
    augmented_sheets[new_name] = df
    del augmented_sheets[old_name]
    idx = sheet_names.index(old_name)
    sheet_names[idx] = new_name
    sheet_pair_list[:] = [(new_name if s == old_name else s, p)
                          for s, p in sheet_pair_list]

print('已将存量因子转化为日变化量:', DIFF_SHEETS)
print('更新后 sheet_names:', sheet_names)


已将存量因子转化为日变化量: {'unit_total': 'unit_chg'}
更新后 sheet_names: ['pct_chg', 'unit_chg', 'netinflow', 'NAV_iopv_discount', 'turn', 'volume_btin']


## 3. 每个 sheet 的标的一览

In [5]:
summary = []
for name in sheet_names:
    df = augmented_sheets[name]
    pairs = get_pair_cols(df)
    summary.append({
        'sheet': name,
        '标的数': len(pairs),
        '行数': len(df),
        '标的列表': pairs
    })

for s in summary:
    print(f"Sheet: {s['sheet']}  标的数: {s['标的数']}  行数: {s['行数']}")
    print(f"  标的: {s['标的列表'][:8]}{'...' if len(s['标的列表']) > 8 else ''}")
    print()

Sheet: pct_chg  标的数: 10  行数: 1689
  标的: ['159977.SZ', '159952.SZ', '159915.SZ', '588050.SH', '588080.SH', '510050.SH', '510100.SH', 'Unnamed: 8']...

Sheet: unit_chg  标的数: 7  行数: 1723
  标的: ['510100.SH', '588050.SH', '159977.SZ', '588080.SH', '510050.SH', '159915.SZ', '159952.SZ']

Sheet: netinflow  标的数: 7  行数: 1689
  标的: ['159977.SZ', '159952.SZ', '159915.SZ', '588050.SH', '588080.SH', '510050.SH', '510100.SH']

Sheet: NAV_iopv_discount  标的数: 7  行数: 1689
  标的: ['159977.SZ', '159952.SZ', '159915.SZ', '588050.SH', '588080.SH', '510050.SH', '510100.SH']

Sheet: turn  标的数: 7  行数: 1689
  标的: ['159977.SZ', '159952.SZ', '159915.SZ', '588050.SH', '588080.SH', '510050.SH', '510100.SH']

Sheet: volume_btin  标的数: 7  行数: 1689
  标的: ['159977.SZ', '159952.SZ', '159915.SZ', '588050.SH', '588080.SH', '510050.SH', '510100.SH']



## 4. 按 sheet × 标的 取单列序列（示例）

In [6]:
def get_factor_series(sheet_name, pair, sheets=None):
    """取指定 sheet、指定标的的 [交易日期, 值] 序列。"""
    sheets = sheets or augmented_sheets
    if sheet_name not in sheets:
        return None
    df = sheets[sheet_name][[DATE_COL, pair]].copy()
    df = df.rename(columns={pair: 'value'})
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    return df.dropna(subset=[DATE_COL, 'value']).sort_values(DATE_COL).reset_index(drop=True)


# 示例：取第一个 sheet 的第一个标的
if sheet_pair_list:
    first_sheet, first_pair = sheet_pair_list[0]
    series = get_factor_series(first_sheet, first_pair)
    print(f'示例: sheet={first_sheet}, 标的={first_pair}')
    print('行数:', len(series))
    display(series.head(10))

示例: sheet=pct_chg, 标的=159977.SZ
行数: 1506


,交易日期,value
0,2019-09-30,-1.272727
1,2019-10-08,-0.736648
2,2019-10-09,0.247372
3,2019-10-10,2.837754
4,2019-10-11,-0.059988
5,2019-10-14,0.720288
6,2019-10-15,-1.132300
7,2019-10-16,-0.060277
8,2019-10-17,0.000000
9,2019-10-18,-0.482509


## 6. Rolling z-score（窗口在 config.ROLLING_ZSCORE_WINDOW 中可调）

对每个因子(sheet)的每个标的列做 **rolling z-score**：  
`z = (x - rolling_mean) / rolling_std`，窗口内 std=0 时置为 0。

In [7]:
ROLLING_WINDOW = getattr(config, 'ROLLING_ZSCORE_WINDOW', 180)
MIN_VALID = getattr(config, 'ROLLING_ZSCORE_MIN_VALID_IN_WINDOW', 180)

def rolling_zscore(series, window):
    """Rolling z-score：仅用窗口内有效值（非空）算均值和标准差，当日有效且窗口内至少 MIN_VALID 个有效值则输出 z，保证连续。"""
    x = pd.to_numeric(series, errors='coerce')
    valid = x.notna()
    x_for_roll = x.where(valid)  # 无效日不参与 rolling 的 mean/std
    roll_mean = x_for_roll.rolling(window=window, min_periods=MIN_VALID).mean()
    roll_std = x_for_roll.rolling(window=window, min_periods=MIN_VALID).std()
    z = (x - roll_mean) / roll_std.where(roll_std > 1e-12)
    z = z.where(valid)  # 仅当日有效才输出
    return z

# augmented_sheets_z: 每个 sheet 的 DataFrame 中，标的列已变为 rolling z-score
augmented_sheets_z = {}
for name in sheet_names:
    df = augmented_sheets[name].copy()
    pairs = get_pair_cols(df)
    for col in pairs:
        if col in df.columns:
            raw = pd.to_numeric(df[col], errors='coerce').ffill()   # ffill 原始数据，填补单日缺失
            df[col] = rolling_zscore(raw, ROLLING_WINDOW)
    augmented_sheets_z[name] = df

print('Rolling 窗口:', ROLLING_WINDOW, '，仅用有效值算 mean/std，窗口内至少', MIN_VALID, '个有效值才输出 z')
print('augmented_sheets_z 已生成，键:', list(augmented_sheets_z.keys()))

Rolling 窗口: 90 ，仅用有效值算 mean/std，窗口内至少 90 个有效值才输出 z
augmented_sheets_z 已生成，键: ['pct_chg', 'unit_chg', 'netinflow', 'NAV_iopv_discount', 'turn', 'volume_btin']


## 7. 相对因子：同一因子内按标的对做差（config.RELATIVE_FACTOR_PAIRS）

对每对 (标的A, 标的B)，在同一 sheet 内计算 **rolling_z(标的A) - rolling_z(标的B)**，形成相对资金净流入类因子。  
列名示例：`510050.SH-510100.SH`。

In [8]:
RELATIVE_PAIRS = getattr(config, 'RELATIVE_FACTOR_PAIRS', [
    ("510050.SH", "510100.SH"),
    ("588080.SH", "588050.SH"),
    ("159915.SZ", "159952.SZ"),
    ("159915.SZ", "159977.SZ"),
])

# relative_factors: { sheet_name: DataFrame }，列 = 交易日期 + 各对 "codeA-codeB"
relative_factors = {}
for name in sheet_names:
    df_z = augmented_sheets_z[name]
    out = df_z[[DATE_COL]].copy()
    for code_a, code_b in RELATIVE_PAIRS:
        if code_a in df_z.columns and code_b in df_z.columns:
            col_name = f"{code_a}-{code_b}"
            out[col_name] = df_z[code_a] - df_z[code_b]
    relative_factors[name] = out

print('相对因子对数:', len(RELATIVE_PAIRS))
print('RELATIVE_FACTOR_PAIRS:', RELATIVE_PAIRS)
for name in list(relative_factors.keys())[:3]:
    print(f'  {name}: 列 {list(relative_factors[name].columns)}')

相对因子对数: 4
RELATIVE_FACTOR_PAIRS: [('510050.SH', '510100.SH'), ('588080.SH', '588050.SH'), ('159915.SZ', '159952.SZ'), ('159915.SZ', '159977.SZ')]
  pct_chg: 列 ['交易日期', '510050.SH-510100.SH', '588080.SH-588050.SH', '159915.SZ-159952.SZ', '159915.SZ-159977.SZ']
  unit_chg: 列 ['交易日期', '510050.SH-510100.SH', '588080.SH-588050.SH', '159915.SZ-159952.SZ', '159915.SZ-159977.SZ']
  netinflow: 列 ['交易日期', '510050.SH-510100.SH', '588080.SH-588050.SH', '159915.SZ-159952.SZ', '159915.SZ-159977.SZ']


## 7b. 每组因子有效值汇报

相对因子中：做差为 NaN（任一侧 rolling 窗口内有 0 或空）的日已剔除，此处汇报每组 (sheet, 标的对) 的有效值数量。

In [9]:
valid_counts = []
for name in sheet_names:
    df = relative_factors[name]
    pair_cols = [c for c in df.columns if c != DATE_COL]
    for col in pair_cols:
        n_valid = df[col].notna().sum()
        valid_counts.append({'sheet': name, '因子(标的对)': col, '有效值': int(n_valid)})
valid_df = pd.DataFrame(valid_counts)
print('每组因子有效值（相对因子：rolling 窗口内无 0/空 且 做差有效 的交易日数）')
display(valid_df)
print('汇总: 共', len(valid_df), '组因子')

每组因子有效值（相对因子：rolling 窗口内无 0/空 且 做差有效 的交易日数）


,sheet,因子(标的对),有效值
0,pct_chg,510050.SH-510100.SH,1415
1,pct_chg,588080.SH-588050.SH,1146
2,pct_chg,159915.SZ-159952.SZ,1600
3,pct_chg,159915.SZ-159977.SZ,1418
4,unit_chg,510050.SH-510100.SH,1466
5,unit_chg,588080.SH-588050.SH,1209
6,unit_chg,159915.SZ-159952.SZ,1633
7,unit_chg,159915.SZ-159977.SZ,1462
8,netinflow,510050.SH-510100.SH,1416
9,netinflow,588080.SH-588050.SH,1147


汇总: 共 24 组因子


## 8. 相对因子汇总（供下游与 y 对齐）

- **augmented_sheets_z**：各 sheet 的 rolling z-score 序列  
- **relative_factors**：各 sheet 的「标的对做差」相对因子，列名如 `510050.SH-510100.SH`

In [10]:
# 示例：netinflow 的 510050.SH-510100.SH 相对因子前几行
if 'netinflow' in relative_factors:
    display(relative_factors['netinflow'].sample(10))

# 输出各 relative 因子时间序列到 factor_build/outputs（每次运行覆盖更新）
_out_dir = os.path.join(os.getcwd(), getattr(config, 'FACTOR_BUILD_OUTPUTS', 'outputs'))
os.makedirs(_out_dir, exist_ok=True)
_path_rf = os.path.join(_out_dir, '02_relative_factors_timeseries.xlsx')
with pd.ExcelWriter(_path_rf, engine='openpyxl') as _w:
    for _name, _df in relative_factors.items():
        _df.to_excel(_w, sheet_name=_name[:31], index=False)
print('已输出:', _path_rf)

,交易日期,510050.SH-510100.SH,588080.SH-588050.SH,159915.SZ-159952.SZ,159915.SZ-159977.SZ
975,2023-01-06,0.923847,7.075017,-0.443878,-0.885756
1541,2025-05-15,0.061858,-0.237053,0.077171,-0.243088
1027,2023-03-28,1.905402,-0.886874,-0.167927,0.295138
399,2020-08-24,0.315969,NaN,0.129474,0.004619
1676,2025-12-01,0.599927,0.169050,0.123472,-0.313482
1158,2023-10-13,-2.965340,-0.493992,0.603230,0.745901
148,2019-08-12,NaN,NaN,-0.091873,NaN
1229,2024-01-23,2.492706,-0.739947,-1.436883,-0.104779
439,2020-10-27,-1.241073,NaN,0.003634,-0.953797
82,2019-05-09,NaN,NaN,NaN,NaN


已输出: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_build/outputs/02_relative_factors_timeseries.xlsx


## 9. Relative factors 大汇总

汇报所有收集到的 relative factor 情况：按 sheet × 标的对 列出有效值、日期范围、均值/标准差。

In [11]:
rows = []
for name in sheet_names:
    df = relative_factors[name]
    pair_cols = [c for c in df.columns if c != DATE_COL]
    for col in pair_cols:
        s = df[col].dropna()
        n = len(s)
        if n == 0:
            date_min = date_max = None
            mean_val = std_val = None
        else:
            dates = df.loc[s.index, DATE_COL]
            date_min = dates.min()
            date_max = dates.max()
            mean_val = round(s.mean(), 4)
            std_val = round(s.std(), 4)
        rows.append({
            'sheet(因子类型)': name,
            '标的对': col,
            '有效值': n,
            '日期起': date_min,
            '日期止': date_max,
            'mean': mean_val,
            'std': std_val
        })
summary_all = pd.DataFrame(rows)

print('========== Relative factors 大汇总 ==========')
print('共', len(summary_all), '组 relative factor（sheet × 标的对）')
print('标的对配置:', RELATIVE_PAIRS)
print('Rolling 窗口:', ROLLING_WINDOW)
print()
display(summary_all)
print()
print('按 sheet 汇总有效值:')
by_sheet = summary_all.groupby('sheet(因子类型)')['有效值'].agg(['sum', 'count']).rename(columns={'sum': '总有效值', 'count': '因子数'})
display(by_sheet)
print('总有效观测数（所有因子有效值之和）:', summary_all['有效值'].sum())

========== Relative factors 大汇总 ==========
共 24 组 relative factor（sheet × 标的对）
标的对配置: [('510050.SH', '510100.SH'), ('588080.SH', '588050.SH'), ('159915.SZ', '159952.SZ'), ('159915.SZ', '159977.SZ')]
Rolling 窗口: 90



,sheet(因子类型),标的对,有效值,日期起,日期止,mean,std
0,pct_chg,510050.SH-510100.SH,1415,2020-02-21,2025-12-17,0.0004,0.1286
1,pct_chg,588080.SH-588050.SH,1146,2021-03-30,2025-12-17,0.0002,0.1323
2,pct_chg,159915.SZ-159952.SZ,1600,2019-05-20,2025-12-17,0.0001,0.1179
3,pct_chg,159915.SZ-159977.SZ,1418,2020-02-18,2025-12-17,0.0002,0.1383
4,unit_chg,510050.SH-510100.SH,1466,2020-01-21,2026-02-05,-0.0746,1.2579
5,unit_chg,588080.SH-588050.SH,1209,2021-02-10,2026-02-05,0.0120,1.2092
6,unit_chg,159915.SZ-159952.SZ,1633,2019-05-21,2026-02-05,0.0029,1.2223
7,unit_chg,159915.SZ-159977.SZ,1462,2020-02-04,2026-02-05,0.0537,1.2907
8,netinflow,510050.SH-510100.SH,1416,2020-02-20,2025-12-17,-0.0401,1.1778
9,netinflow,588080.SH-588050.SH,1147,2021-03-29,2025-12-17,0.0193,1.1596



按 sheet 汇总有效值:


,总有效值,因子数
sheet(因子类型),,
NAV_iopv_discount,5582,4
netinflow,5582,4
pct_chg,5579,4
turn,5582,4
unit_chg,5770,4
volume_btin,5582,4


总有效观测数（所有因子有效值之和）: 33677


## 5. 汇总变量（供后续 notebook 使用）

- **augmented_sheets**：{ sheet_name: DataFrame }，已规整日期列
- **sheet_pair_list**：[(sheet, 标的), ...] 全部组合
- **get_factor_series(sheet_name, pair)**：取某 sheet 某标的的时序

In [12]:
print('augmented_sheets 键:', list(augmented_sheets.keys()))
print('sheet_pair_list 前 5 个:', sheet_pair_list[:5])

augmented_sheets 键: ['pct_chg', 'netinflow', 'NAV_iopv_discount', 'turn', 'volume_btin', 'unit_chg']
sheet_pair_list 前 5 个: [('pct_chg', '159977.SZ'), ('pct_chg', '159952.SZ'), ('pct_chg', '159915.SZ'), ('pct_chg', '588050.SH'), ('pct_chg', '588080.SH')]
